# Bài 2: Thu thập dữ liệu quốc gia và vùng lãnh thổ

**Họ tên:** Đoàn Phúc Nguyên
**MSSV:** 24521174

## 1. Mục tiêu
Thu thập tối thiểu 100 bản ghi từ một website,
lưu dưới dạng CSV và thống kê cơ bản về dữ liệu.

## 2. Nguồn dữ liệu
https://www.scrapethissite.com/pages/simple/

Mỗi bản ghi tương ứng một quốc gia hoặc vùng lãnh thổ,
gồm: tên, thủ đô, dân số và diện tích (km²).

In [1]:
%pip -q install requests beautifulsoup4 pandas

In [8]:
%%writefile thu_thap_quoc_gia.py

import requests
from bs4 import BeautifulSoup
import pandas as pd

def thu_thap_du_lieu(url):

    response = requests.get(url, timeout=30)
    response.raise_for_status()
    response.encoding = "utf-8"

    soup = BeautifulSoup(response.text, "html.parser")
    cac_quoc_gia = soup.select("div.country")

    if not cac_quoc_gia:
        raise ValueError("Không tìm thấy dữ liệu trên website này.")

    du_lieu = []

    for quoc_gia in cac_quoc_gia:
        ten = quoc_gia.select_one(".country-name").get_text(strip=True)
        thu_do = quoc_gia.select_one(".country-capital").get_text(strip=True)
        dan_so = quoc_gia.select_one(".country-population").get_text(strip=True)
        dien_tich = quoc_gia.select_one(".country-area").get_text(strip=True)

        du_lieu.append({
            "ten_quoc_gia": ten,
            "thu_do": thu_do,
            "dan_so": int(dan_so),
            "dien_tich_km2": float(dien_tich)
        })

    return pd.DataFrame(du_lieu)

def tien_xu_li(bang_du_lieu):

    bang_sach = bang_du_lieu.copy()

    bang_sach["thu_do"] = bang_sach["thu_do"].replace({
        "None": pd.NA,
        "": pd.NA
    })

    bang_sach = bang_sach.drop_duplicates().reset_index(drop=True)

    return bang_sach

def thong_ke_du_lieu(bang_du_lieu):

    return bang_du_lieu[["dan_so", "dien_tich_km2"]].describe()

def luu_du_lieu(bang_du_lieu, ten_file):

    if len(bang_du_lieu) < 100:
        raise ValueError("Chưa đủ tối thiểu 100 điểm dữ liệu theo thầy yêu cầu.")

    bang_du_lieu.to_csv(
        ten_file,
        index=False,
        encoding="utf-8-sig"
    )


if __name__ == "__main__":
    url = "https://www.scrapethissite.com/pages/simple/"

    bang_ban_dau = thu_thap_du_lieu(url)
    bang_du_lieu = tien_xu_li(bang_ban_dau)

    luu_du_lieu(bang_du_lieu, "quoc_gia.csv")

    print("Số bản ghi:", len(bang_du_lieu))
    print(thong_ke_du_lieu(bang_du_lieu))

Writing thu_thap_quoc_gia.py


In [11]:
import importlib
import thu_thap_quoc_gia

importlib.reload(thu_thap_quoc_gia)

from thu_thap_quoc_gia import (
    thu_thap_du_lieu,
    tien_xu_li,
    thong_ke_du_lieu,
    luu_du_lieu
)

from IPython.display import display

url = "https://www.scrapethissite.com/pages/simple/"

bang_ban_dau = thu_thap_du_lieu(url)
bang_du_lieu = tien_xu_li(bang_ban_dau)

bang_thong_ke = thong_ke_du_lieu(bang_du_lieu)
luu_du_lieu(bang_du_lieu, "quoc_gia.csv")

print("Số dòng ban đầu:", len(bang_ban_dau))
print("Số dòng sau tiền xử lý:", len(bang_du_lieu))
print("Số cột:", bang_du_lieu.shape[1])

print("\nSố ô thiếu theo từng cột:")
print(bang_du_lieu.isna().sum())

print("\n10 dòng đầu tiên:")
display(bang_du_lieu.head(10))


Số dòng ban đầu: 250
Số dòng sau tiền xử lý: 250
Số cột: 4

Số ô thiếu theo từng cột:
ten_quoc_gia     0
thu_do           8
dan_so           0
dien_tich_km2    0
dtype: int64

10 dòng đầu tiên:


,ten_quoc_gia,thu_do,dan_so,dien_tich_km2
0,Andorra,Andorra la Vella,84000,468.0
1,United Arab Emirates,Abu Dhabi,4975593,82880.0
2,Afghanistan,Kabul,29121286,647500.0
3,Antigua and Barbuda,St. John's,86754,443.0
4,Anguilla,The Valley,13254,102.0
5,Albania,Tirana,2986952,28748.0
6,Armenia,Yerevan,2968000,29800.0
7,Angola,Luanda,13068161,1246700.0
8,Antarctica,<NA>,0,14000000.0
9,Argentina,Buenos Aires,41343201,2766890.0


## 3. Phương pháp thu thập và tiền xử lý
### Công cụ sử dụng
- requests: tải nội dung HTML của website.
- BeautifulSoup: trích xuất thông tin từ HTML.
- pandas: xử lý bảng dữ liệu, thống kê và lưu CSV.

### Cấu trúc dữ liệu
Mỗi bản ghi tương ứng một quốc gia hoặc vùng lãnh thổ,
gồm 4 thuộc tính:

| Thuộc tính | Ý nghĩa |
|---|---|
| ten_quoc_gia | Tên quốc gia hoặc vùng lãnh thổ |
| thu_do | Thủ đô được ghi trên website |
| dan_so | Dân số, đơn vị người |
| dien_tich_km2 | Diện tích, đơn vị km² |

### Tổ chức chương trình
Source code được tổ chức trong module thu_thap_quoc_gia.py,
gồm các hàm:
- thu_thap_du_lieu: tải trang và lấy dữ liệu.
- tien_xu_li: chuẩn hóa giá trị thiếu, loại dòng trùng.
- thong_ke_du_lieu: tính thống kê dân số và diện tích.
- luu_du_lieu: lưu dữ liệu thành file CSV.

### Tiền xử lý
Dân số được chuyển thành số nguyên, diện tích thành số thực.
Thông tin thủ đô ghi là "None" hoặc chuỗi rỗng được chuyển
thành giá trị thiếu. Các dòng trùng hoàn toàn được loại bỏ.

## 4. Kết quả và thống kê cơ bản

In [12]:
print("Số bản ghi:", len(bang_du_lieu))
print("Số thuộc tính:", bang_du_lieu.shape[1])

so_dong_trung = len(bang_ban_dau) - len(bang_du_lieu)
print("Số dòng trùng ở mọi cột đã loại bỏ:", so_dong_trung)

print("\nSố ô thiếu theo từng thuộc tính:")
print(bang_du_lieu.isna().sum())

print("\nThống kê dân số và diện tích:")
display(bang_thong_ke.round(2))

from google.colab import files
files.download("quoc_gia.csv")

Số bản ghi: 250
Số thuộc tính: 4
Số dòng trùng ở mọi cột đã loại bỏ: 0

Số ô thiếu theo từng thuộc tính:
ten_quoc_gia     0
thu_do           8
dan_so           0
dien_tich_km2    0
dtype: int64

Thống kê dân số và diện tích:


,dan_so,dien_tich_km2
count,2.500000e+02,250.00
mean,2.744568e+07,599636.92
std,1.168626e+08,1911821.26
min,0.000000e+00,0.00
25%,1.798562e+05,1174.75
50%,4.288138e+06,64894.50
75%,1.542062e+07,372631.50
max,1.330044e+09,17100000.00


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 5. Nhận xét và giới hạn

Bộ dữ liệu sau tiền xử lý đã đáp ứng yêu cầu của thầy là tối thiểu
100 điểm dữ liệu và được lưu trong file quoc_gia.csv.

Bảng thống kê trình bày số giá trị không thiếu, trung bình,
độ lệch chuẩn mẫu, nhỏ nhất, các phân vị và lớn nhất
của dân số và diện tích.

Dữ liệu gồm cả quốc gia và vùng lãnh thổ.